# Day 19 - Structured data (lesson 7)

Goal: get clean JSON back from Claude with **prefill + stop sequence**, then parse it.

- Prefill: start the assistant turn with the opening code fence so Claude continues straight into the JSON.
- Stop sequence: the closing fence ends generation, so nothing but the JSON comes back.

> Prefill is rejected (HTTP 400) on the newest models (Claude 4.6+ and the 5.x family).
> This notebook uses `claude-haiku-4-5`, which still accepts it, so the lesson runs as taught.
> On newer models use structured outputs (`output_config.format`) or a prompt instruction instead.

In [ ]:
import json
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()  # reads ANTHROPIC_API_KEY from .env
client = Anthropic()
MODEL = "claude-haiku-4-5"

In [ ]:
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})


def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})


def chat(messages, system=None, temperature=0.0, stop_sequences=None):
    params = {
        "model": MODEL,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }
    if system:
        params["system"] = system  # omit when empty: None is rejected
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    message = client.messages.create(**params)
    return message.content[0].text

In [ ]:
messages = []
add_user_message(
    messages,
    "Write a JSON object describing a reminder with keys: title, datetime (ISO 8601), notes.",
)
add_assistant_message(messages, "```json")  # prefill

text = chat(messages, stop_sequences=["```"])
print("raw :", repr(text))

reminder = json.loads(text.strip())
print("dict:", reminder)
print("title:", reminder["title"])

## Check

- Why does the closing fence work as a stop sequence?
- What happens if you remove the prefill? (Run it and look at the raw text.)
- Where would a code-based grader (`validate_json`) plug in for the eval lessons?